**Indicador:** Porcentaje de nacimientos de mujeres de 19 años o menos en la Ciudad de México.

**Fuente:** INEGI, Estadística de Nacimientos Registrados (ENR). | **Universo:** Nacimientos registrados de madres de 19 años o menos residentes en la Ciudad de México.

**Método:** Se calcula el porcentaje de nacimientos registrados de madres de 19 años o menos respecto del total de nacimientos registrados de madres residentes en la Ciudad de México durante el mismo año. Para ello, se procesan los microdatos de la ENR, identificando los nacimientos según la entidad de residencia de la madre y su edad al momento del parto. El resultado se expresa como porcentaje del total de nacimientos registrados en la entidad.

La prospectiva a 2047 se construye a partir de la serie histórica disponible, tomando como línea base el último año observado. Se utiliza un modelo de crecimiento exponencial basado en tasas de variación media anual (TVMA), derivadas de las metas normativas definidas para los tres escenarios prospectivos.

In [1]:
# limpiar entorno de trabajo
rm(list = ls()) 

In [2]:
# Librerías 
if(!require('pacman')) install.packages('pacman')
pacman::p_load(tidyverse, showtext, srvyr,httr,utils,foreign)
remotes::install_github("cpyp-ipdp/funcionespgd", force = TRUE)
library(funcionespgd)
font_add_google("Poppins", "pop")
showtext_auto()

Cargando paquete requerido: pacman




rlang    (1.1.7 -> 1.3.0) [CRAN]
glue     (1.8.0 -> 1.8.1) [CRAN]
cli      (3.6.5 -> 3.6.6) [CRAN]
vctrs    (0.7.1 -> 0.7.3) [CRAN]
magrittr (2.0.4 -> 2.0.5) [CRAN]
S7       (0.2.1 -> 0.2.2) [CRAN]
dplyr    (1.2.0 -> 1.2.1) [CRAN]
ggplot2  (4.0.2 -> 4.0.3) [CRAN]


Installing 8 packages: rlang, glue, cli, vctrs, magrittr, S7, dplyr, ggplot2

Warning message:
"packages 'dplyr', 'ggplot2' are in use and will not be installed"
Installing packages into 'C:/Users/IPDP 2026/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'rlang' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'rlang'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problema al copiar C:\Users\IPDP 2026\AppData\Local\R\win-library\4.5\00LOCK\rlang\libs\x64\rlang.dll  a C:\Users\IPDP 2026\AppData\Local\R\win-library\4.5\rlang\libs\x64\rlang.dll: Permission denied"
Warning message:
"restored 'rlang'"


package 'glue' successfully unpacked and MD5 sums checked


... (9 outputs left out)


The downloaded binary packages are in
	C:\Users\IPDP 2026\AppData\Local\Temp\RtmpUTjjBV\downloaded_packages
── R CMD build ─────────────────────────────────────────────────────────────────



Please download and install Rtools 4.5 from https://cran.r-project.org/bin/windows/Rtools/.



* checking for file 'C:\Users\IPDP 2026\AppData\Local\Temp\RtmpUTjjBV\remotes5740687f2e14\cpyp-ipdp-funcionespgd-6fd64b9/DESCRIPTION' ... OK
* preparing 'funcionespgd':
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted 'LazyData' from DESCRIPTION
* building 'funcionespgd_0.1.0.tar.gz'



Installing package into 'C:/Users/IPDP 2026/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



In [3]:
dir.create("nacimientos", showWarnings = FALSE)
url_1_default <- "https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/"
#url_1_2016    <- "https://www.inegi.org.mx/contenidos/programas/natalidad/Microdatos/"  # Nota: 'M' mayúscula

url_2 <- "natalidad_base_datos_"
years <- 2018:2025

root_dir <- "nacimientos"

In [4]:
## DESCARGA DE BASES

options(timeout=600)

for (yr in years) {
  fin <- "_dbf.zip"
  base_url <- if (yr == 2016) url_1_2016 else url_1_default
  url <- paste0(base_url, yr, "/", url_2, yr, fin)

  dest_zip <- file.path(root_dir, paste0(yr, fin))
  message("Descargando: ", url)

  dl_ok <- tryCatch({
    utils::download.file(url, destfile = dest_zip, mode = "wb", quiet = TRUE)
    TRUE
  }, error = function(e) {
    message("Error en la descarga (", yr, "): ", conditionMessage(e))
    FALSE
  })
  if (!dl_ok || !file.exists(dest_zip)) next

  # Registrar estado previo para identificar qué archivos aparecen tras unzip
  before_files <- list.files(root_dir, recursive = TRUE, full.names = TRUE)

  # CÓDIGO CORREGIDO PARA CAPTURAR ADVERTENCIAS (WARNINGS)
  uz_ok <- tryCatch({
    # Al agregar warning = function(w)... obligamos a tratar el aviso como un fallo
    utils::unzip(dest_zip, exdir = root_dir)
    TRUE
  }, warning = function(w) {
    message("   Advertencia en la descompresión (", yr, "): ", conditionMessage(w))
    FALSE
  }, error = function(e) {
    message("   Error crítico en la descompresión (", yr, "): ", conditionMessage(e))
    FALSE
  })

  if (!uz_ok) {
    message(" Conservando ZIP por fallo en descompresión: ", dest_zip)
    next
  }

  # Identificar nuevos archivos creados por este unzip
  
 after_files <- list.files(root_dir, recursive = TRUE, full.names = TRUE)
  new_paths <- setdiff(after_files, before_files)

  # Filtrar solo archivos (no directorios) y que no sean .zip

  new_files <- new_paths[file.exists(new_paths)]
  new_files <- new_files[!dir.exists(new_files)]
  new_files <- new_files[!grepl("\\.zip$", new_files, ignore.case = TRUE)]

  # Mover al root y resolver colisiones

  for (src in new_files) {
    tgt_name <- basename(src)
    tgt <- file.path(root_dir, tgt_name)

    if (normalizePath(dirname(src)) == normalizePath(root_dir)) {
      # Ya está en root, no mover
      next
    }

    if (file.exists(tgt)) {
      # Si hay colisión, insertamos _<año> antes de la extensión
      base <- tools::file_path_sans_ext(tgt_name)
      ext  <- tools::file_ext(tgt_name)
      if (nzchar(ext)) {
        tgt <- file.path(root_dir, paste0(base, "_", yr, ".", ext))
      } else {
        tgt <- file.path(root_dir, paste0(base, "_", yr))
      }
    }

    # Crear carpeta destino si no existe 

    dir.create(dirname(tgt), recursive = TRUE, showWarnings = FALSE)

    ok_move <- tryCatch({
      file.rename(src, tgt)
    }, warning = function(w) FALSE, error = function(e) FALSE)

    if (!ok_move) {
      # Si file.rename falla (p.ej. entre discos), usar copia + borrado
      ok_copy <- tryCatch({
        file.copy(src, tgt, overwrite = FALSE)
      }, warning = function(w) FALSE, error = function(e) FALSE)
      if (isTRUE(ok_copy)) unlink(src)
    }
  }

  # Limpiar directorios vacíos creados durante el unzip
  # (solo dentro de nacimientos/)

  dirs_created <- unique(dirname(setdiff(after_files, before_files)))
  dirs_created <- dirs_created[dirs_created != root_dir]
  # Orden inverso para borrar de adentro hacia afuera
  dirs_created <- dirs_created[order(nchar(dirs_created), decreasing = TRUE)]
  for (d in dirs_created) {
    if (dir.exists(d)) {
      # Si está vacío, lo borra; si no, lo deja
      try(unlink(d, recursive = FALSE, force = TRUE), silent = TRUE)
      # Intento extra: si sigue habiendo subcarpetas vacías, intentar recursivo pero seguro
      if (dir.exists(d) && length(list.files(d, recursive = TRUE)) == 0) {
        try(unlink(d, recursive = TRUE, force = TRUE), silent = TRUE)
      }
    }
  }

  # Borrar ZIP

  rm_ok <- tryCatch({
    file.remove(dest_zip)
  }, error = function(e) {
    message(" Error al eliminar el ZIP (", yr, "): ", conditionMessage(e))
    FALSE
  })
  if (isTRUE(rm_ok)) {
    message(" Año ", yr, ": extraído en ", root_dir, " y ZIP eliminado.")
  } else {
    message(" Año ", yr, ": extraído en ", root_dir, ". (ZIP conservado)")
  }
}

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2018/natalidad_base_datos_2018_dbf.zip

 Año 2018: extraído en nacimientos y ZIP eliminado.

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2019/natalidad_base_datos_2019_dbf.zip

 Año 2019: extraído en nacimientos y ZIP eliminado.

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2020/natalidad_base_datos_2020_dbf.zip

 Año 2020: extraído en nacimientos y ZIP eliminado.

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2021/natalidad_base_datos_2021_dbf.zip

 Año 2021: extraído en nacimientos y ZIP eliminado.

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2022/natalidad_base_datos_2022_dbf.zip

 Año 2022: extraído en nacimientos y ZIP eliminado.

Descargando: https://www.inegi.org.mx/contenidos/programas/natalidad/microdatos/2023/natalidad_base_datos_2023_dbf.zip

 Año 2023: extr

In [5]:
##Identificar los archivos que contengan Nacimientos

lista_archivos <- list.files(path = "nacimientos", full.names = TRUE)

archivos_nac <- lista_archivos %>%
  str_subset("/[Nn][Aa][Cc][Ii][Mm]")   
archivos_nac

[1] "nacimientos/NACIM00_2000.dbf" "nacimientos/NACIM01_2001.dbf"
 [3] "nacimientos/NACIM02_2002.dbf" "nacimientos/NACIM03_2003.dbf"
 [5] "nacimientos/NACIM04_2004.dbf" "nacimientos/NACIM05_2005.dbf"
 [7] "nacimientos/NACIM06_2006.dbf" "nacimientos/NACIM07_2007.dbf"
 [9] "nacimientos/NACIM08_2008.dbf" "nacimientos/NACIM09_2009.dbf"
[11] "nacimientos/NACIM10_2010.dbf" "nacimientos/NACIM11_2011.dbf"
[13] "nacimientos/NACIM12_2012.dbf" "nacimientos/NACIM13_2013.dbf"
[15] "nacimientos/NACIM14_2014.dbf" "nacimientos/NACIM15_2015.dbf"
[17] "nacimientos/NACIM16_2016.DBF" "nacimientos/NACIM17_2017.DBF"
[19] "nacimientos/NACIM18.DBF"      "nacimientos/NACIM18_2018.DBF"
[21] "nacimientos/NACIM19.DBF"      "nacimientos/NACIM19_2019.DBF"
[23] "nacimientos/NACIM20.DBF"      "nacimientos/NACIM20_2020.DBF"
[25] "nacimientos/nacim21.dbf"      "nacimientos/nacim21_2021.dbf"
[27] "nacimientos/nacim22.dbf"      "nacimientos/nacim22_2022.dbf"
[29] "nacimientos/nacim23.dbf"      "nacimientos/nacim23_2023.dbf"
[31] "nacimientos/nacim24.dbf"      "nacimientos/nacim24_2024.dbf"
[33] "nacimientos/nacim25.dbf"      "nacimientos/NACIM85.dbf"     
[35] "nacimientos/NACIM86.dbf"      "nacimientos/NACIM87_1987.dbf"
[37] "nacimientos/NACIM88_1988.dbf" "nacimientos/NACIM89_1989.dbf"
[39] "nacimientos/NACIM90_1990.dbf" "nacimientos/NACIM91_1991.dbf"
[41] "nacimientos/NACIM92_1992.dbf" "nacimientos/NACIM93_1993.dbf"
[43] "nacimientos/NACIM94_1994.dbf" "nacimientos/NACIM95_1995.dbf"
[45] "nacimientos/NACIM96_1996.dbf" "nacimientos/NACIM97_1997.dbf"
[47] "nacimientos/NACIM98_1998.dbf" "nacimientos/NACIM99_1999.dbf"

In [6]:
## CALCULO DEL INDICADOR

embarazo_adolescente <- function(path, entidad = 9) {
  # 1) extraer año de los dos últimos dígitos del nombre
  suf2 <- str_match(basename(path), "(?i)nacim(\\d{2})")[,2] |> as.integer()
  year <- ifelse(suf2 >= 85, 1900 + suf2, 2000 + suf2)

  # 2) leer base y limpiar
  df <- read.dbf(path, as.is = TRUE) |>
    janitor::clean_names() |>
     mutate(ent_resid = as.integer(ent_resid))

  # 3) filtrar por entidad (si se pide)
  if (!is.null(entidad)) {
    df <- filter(df, ent_resid == entidad)
  }

  # 4) calcular métricas
  total_nac      <- nrow(df)
  nac_menores19  <- sum(df$edad_madn <= 19, na.rm = TRUE)
  pct  <- 100 * nac_menores19 / total_nac

  tibble(
    year,
    total_nac,
    nac_menores19,
    pct
  )
}

In [7]:
nac_embarazo_adolescente <- map_dfr(archivos_nac, embarazo_adolescente, entidad = 9)%>%
  arrange(year)

nac_embarazo_adolescente

year,total_nac,nac_menores19,pct
<dbl>,<int>,<int>,<dbl>
1985,262424,34947,13.316998
1986,258739,35260,13.627632
1987,267370,42145,15.762801
1988,255531,39747,15.554669
1989,247752,38480,15.531661
1990,233828,36016,15.402775
1991,222105,35184,15.841156
1992,224702,31947,14.217497
1993,220978,31487,14.248930


In [8]:
# Observados
observados <- nac_embarazo_adolescente %>%

  transmute(
    periodo = year,
    valor = pct
  )

In [9]:
# Parámetros base
anio_base <- max(observados$periodo)

linea_base <- observados %>%

  filter(periodo == anio_base) %>%

  pull(valor)

anios <- min(observados$periodo):2047

n_periodos <- anios - anio_base

In [13]:
# Metas finales
meta_transformador <- 0.52

meta_deseable <- 1.98

meta_inercial <- 4

# TCMA
tcma_transformador <- (meta_transformador / linea_base)^(1/(2047 - anio_base)) - 1

tcma_deseable <- (meta_deseable / linea_base)^(1/(2045 - anio_base)) - 1

tcma_inercial <- (meta_inercial / linea_base)^(1/(2047 - anio_base)) - 1

In [14]:
# Serie completa 
tabla_embarazo_adolescente <- tibble(periodo = anios) %>%

  mutate(
    
    #ESCENARIO DESEABLE
    deseable = case_when(

      periodo <= anio_base ~ observados$valor[match(periodo, observados$periodo)],

      TRUE ~ linea_base * ((1 + tcma_deseable)^(periodo - anio_base))
    ),
    

    #ESCENARIO TRANSFORMADOR
    transformador = case_when(

      periodo <= anio_base ~ observados$valor[match(periodo, observados$periodo)],

      TRUE ~ linea_base * ((1 + tcma_transformador)^(periodo - anio_base))
    ),
    
    #ESCENARIO INERCIAL
    inercial = case_when(

      periodo <= anio_base ~ observados$valor[match(periodo, observados$periodo)],

      TRUE ~ linea_base * ((1 + tcma_inercial)^(periodo - anio_base))
    )
  ) %>%
  
  arrange(periodo) %>%
  
  mutate(
    across(c(deseable, transformador, inercial), ~ round(., 2))
  )

  #TABLA FINAL
  
  tabla_embarazo_adolescente

periodo,deseable,transformador,inercial
<int>,<dbl>,<dbl>,<dbl>
1985,13.32,13.32,13.32
1986,13.63,13.63,13.63
1987,15.76,15.76,15.76
1988,15.55,15.55,15.55
1989,15.53,15.53,15.53
1990,15.40,15.40,15.40
1991,15.84,15.84,15.84
1992,14.22,14.22,14.22
1993,14.25,14.25,14.25


In [15]:
# EXPORTAR A EXCEL
write.csv(
    tabla_embarazo_adolescente,
  "EMBARAZO_MENOR19.csv",
  row.names = FALSE
)